# 0. Imports y Configuración
En esta sección se cargan las librerías necesarias y se definen las rutas y parámetros generales del procesamiento, como el video de entrada, la cantidad de frames por segundo, el intervalo del video y las carpetas de salida.

In [13]:
import math
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyiqa
import seaborn as sns
import torch

from scipy import stats
from skimage.restoration import estimate_sigma
from tqdm import tqdm
from itertools import islice
from matplotlib.widgets import RectangleSelector

In [25]:
# Ruta del video que se desea procesar
RUTA_VIDEO = Path("Videos") / "video30min-11to22.mp4"

# Cantidad de imágenes que se extraerán por cada segundo de video
FRAMES_POR_SEGUNDO = 5.0

# Intervalo opcional del video
# None significa comenzar desde el inicio o terminar al final
SEGUNDO_INICIO = None
SEGUNDO_FIN = None

# Compresión PNG entre 0 y 9
COMPRESION_PNG = 3

# Carpeta donde se guardarán los frames
CARPETA_FRAMES = RUTA_VIDEO.parent / RUTA_VIDEO.stem / "frames"

# Semilla para obtener el mismo muestreo en cada ejecución
SEMILLA = 42

# Dispositivo para calcular NIQE y BRISQUE
DISPOSITIVO = "cuda" if torch.cuda.is_available() else "cpu"

# Carpeta donde se guardarán métricas, resúmenes y gráficas
CARPETA_METRICAS = RUTA_VIDEO.parent / RUTA_VIDEO.stem / "metricas"

# Archivo Excel donde se compararán todas las etapas del proyecto
RUTA_EXCEL_RESUMEN = CARPETA_METRICAS / "resumen.xlsx"

# Cantidad de píxeles tomados de cada frame para graficar el ruido
PIXELES_RUIDO_POR_FRAME = 100

# Carpeta donde se guardarán las máscaras del crosshair
CARPETA_MASKS_CROSSHAIR = RUTA_VIDEO.parent / RUTA_VIDEO.stem / "masks_crosshair"

# Porcentaje central de la imagen donde se buscará el crosshair
PORCENTAJE_ROI_CROSSHAIR = 0.6

# Límites HSV del verde fluorescente principal
HSV_VERDE_FUERTE_MIN = (38, 115, 65)
HSV_VERDE_FUERTE_MAX = (88, 255, 255)

# Diferencia mínima del verde fuerte respecto al rojo y al azul
DOMINANCIA_VERDE_FUERTE = (18, 14)

# Límites HSV de los bordes semitransparentes
HSV_VERDE_BORDE_MIN = (34, 55, 45)
HSV_VERDE_BORDE_MAX = (94, 255, 255)

# Diferencia mínima del verde débil respecto al rojo y al azul
DOMINANCIA_VERDE_BORDE = (8, 6)

# Cantidad de píxeles que pueden crecer los bordes desde el verde fuerte
RADIO_BORDES_CROSSHAIR = 2

# Fracción de la dimensión mínima de la ROI usada para expandir la máscara
FRACCION_DILATACION_CROSSHAIR = 0.0035

# Temporal
CARPETA_DIAGNOSTICOS = CARPETA_MASKS_CROSSHAIR.parent / "diagnosticos_crosshair"
GENERAR_DIAGNOSTICOS = True
CANTIDAD_FRAMES_DIAGNOSTICO = 50

# 1. Extracción Frames

En esta sección se extrae una cantidad definida de frames por segundo, ya sea del video completo o de un fragmento específico. Cada frame se guarda en formato PNG con un nombre consecutivo.


In [3]:
captura = cv2.VideoCapture(str(RUTA_VIDEO))  # Abre el video

fps_video = captura.get(cv2.CAP_PROP_FPS)  # Obtiene los FPS originales
total_frames = captura.get(cv2.CAP_PROP_FRAME_COUNT)  # Obtiene el total de frames
duracion = total_frames / fps_video  # Calcula la duración en segundos

inicio = 0.0 if SEGUNDO_INICIO is None else SEGUNDO_INICIO  # Define el segundo inicial
fin = duracion if SEGUNDO_FIN is None else SEGUNDO_FIN  # Define el segundo final

cantidad_salida = math.ceil((fin - inicio) * FRAMES_POR_SEGUNDO)  # Calcula la cantidad de frames de salida
intervalo = 1 / FRAMES_POR_SEGUNDO  # Calcula los segundos entre cada frame
cantidad_digitos = len(str(cantidad_salida - 1))  # Calcula los dígitos según el último índice

CARPETA_FRAMES.mkdir(parents=True, exist_ok=True)  # Crea la carpeta de salida

for indice in tqdm(range(cantidad_salida), desc="Extrayendo frames", unit="frame"):
    segundo = inicio + indice * intervalo  # Calcula el momento que se extraerá

    captura.set(cv2.CAP_PROP_POS_MSEC, segundo * 1000)  # Mueve el video al momento calculado
    lectura_correcta, frame = captura.read()  # Lee el frame correspondiente

    if not lectura_correcta:
        captura.release()  # Cierra el video antes de detener la ejecución
        raise RuntimeError(f"No se pudo leer el segundo {segundo:.2f}")

    nombre = f"{indice:0{cantidad_digitos}d}.png"  # Crea el nombre consecutivo
    ruta_salida = CARPETA_FRAMES / nombre  # Construye la ruta completa

    cv2.imwrite(str(ruta_salida), frame, [cv2.IMWRITE_PNG_COMPRESSION, COMPRESION_PNG])  # Guarda el frame

captura.release()  # Cierra el video y libera los recursos

print(f"Extracción terminada: {cantidad_salida} frames guardados en {CARPETA_FRAMES}")

Extrayendo frames: 100%|██████████| 3300/3300 [21:03<00:00,  2.61frame/s]

Extracción terminada: 3300 frames guardados en Videos/video30min-11to22/Frames


# 2. Caracterización Calidad

Se calcularán métricas no referenciales para establecer la calidad inicial de los frames antes del procesamiento.

- **NIQE (↓):** Evalúa la calidad general del frame a partir de características propias de imágenes naturales; valores menores suelen indicar mejor calidad.

- **BRISQUE (↓):** Estima la degradación visual mediante características espaciales de la imagen; valores menores suelen indicar mejor calidad.

- **Ruido Estimado $\hat{\sigma}$ (↓):** Aproxima la intensidad del ruido mediante componentes de alta frecuencia; valores menores indican menos ruido estimado.

- **SNR Estimado (↑):** Compara la potencia de una versión suavizada del frame con la potencia de la diferencia respecto al original, mediante $\mathrm{SNR}=10\log_{10}(P_{\mathrm{señal}}/P_{\mathrm{ruido}})$; valores mayores indican una mejor relación señal-ruido.

- **Curtosis:** Mide la presencia de valores extremos en la diferencia entre el frame original y su versión suavizada; valores altos pueden ser compatibles con ruido impulsivo.

La diferencia se calcula como $R=I-\hat{I}$, donde $I$ es el frame original y $\hat{I}$ es su versión suavizada. Esta diferencia se utiliza como una aproximación del ruido, aunque también puede contener bordes y texturas.

In [ ]:
ETAPA = "baseline"

CARPETA_METRICAS.mkdir(parents=True, exist_ok=True)  # Crea la carpeta de métricas
ruta_excel_metricas = CARPETA_METRICAS / f"metricas_{ETAPA}.xlsx"  # Define el Excel detallado de la etapa

rutas_frames = sorted(ruta for ruta in CARPETA_FRAMES.iterdir() if ruta.suffix.lower() == ".png")  # Obtiene los frames en orden
metrica_niqe = pyiqa.create_metric("niqe", device=DISPOSITIVO)  # Prepara NIQE
metrica_brisque = pyiqa.create_metric("brisque", device=DISPOSITIVO)  # Prepara BRISQUE
generador = np.random.default_rng(SEMILLA)  # Permite repetir la misma selección de píxeles

resultados = []
muestras_ruido = []
muestras_intensidad = []

with torch.inference_mode():  # Evita cálculos de entrenamiento innecesarios
    for ruta_frame in tqdm(rutas_frames, desc="Calculando métricas", unit="frame"):
        frame_bgr = cv2.imread(str(ruta_frame), cv2.IMREAD_COLOR)  # Lee el frame

        if frame_bgr is not None:
            gris = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2GRAY).astype(np.float32)  # Convierte el frame a gris
            niqe = float(metrica_niqe(str(ruta_frame)).item())  # Calcula NIQE
            brisque = float(metrica_brisque(str(ruta_frame)).item())  # Calcula BRISQUE
            ruido_estimado = float(estimate_sigma(gris / 255.0, channel_axis=None, average_sigmas=True) * 255.0)  # Estima la intensidad del ruido

            frame_suavizado = cv2.GaussianBlur(gris, (0, 0), sigmaX=1.5, sigmaY=1.5)  # Aproxima la parte principal de la imagen
            ruido_aproximado = gris - frame_suavizado  # Calcula la diferencia usada para aproximar el ruido
            ruido_vector = ruido_aproximado.ravel()  # Organiza los valores del ruido en una sola dimensión

            potencia_senal = float(np.mean(np.square(frame_suavizado)))  # Calcula la potencia de la señal aproximada
            potencia_ruido = float(np.mean(np.square(ruido_vector)))  # Calcula la potencia del ruido aproximado
            potencia_ruido = max(potencia_ruido, np.finfo(float).eps)  # Evita dividir entre cero
            snr_estimado = float(10.0 * np.log10(potencia_senal / potencia_ruido))  # Calcula el SNR estimado en decibelios
            curtosis = float(stats.kurtosis(ruido_vector, fisher=True, bias=False))  # Mide la presencia de valores extremos
            intensidad_media = float(gris.mean())  # Calcula el brillo promedio del frame
            contraste = float(gris.std())  # Calcula la dispersión de intensidades usada como medida de contraste

            cantidad_pixeles = min(PIXELES_RUIDO_POR_FRAME, gris.size)  # Limita los píxeles usados en las gráficas
            indices_ruido = generador.choice(ruido_vector.size, size=cantidad_pixeles, replace=False)  # Selecciona píxeles de ruido sin repetir
            indices_intensidad = generador.choice(gris.size, size=cantidad_pixeles, replace=False)  # Selecciona píxeles de intensidad sin repetir
            muestras_ruido.append(ruido_vector[indices_ruido])  # Guarda la muestra de ruido del frame
            muestras_intensidad.append(gris.ravel()[indices_intensidad])  # Guarda la muestra de intensidades del frame

            indice_frame = int(ruta_frame.stem)
            segundo_video = (0.0 if SEGUNDO_INICIO is None else SEGUNDO_INICIO) + indice_frame / FRAMES_POR_SEGUNDO  # Calcula el tiempo correspondiente al frame
            resultados.append({"frame": ruta_frame.name, "segundo_video": segundo_video, "niqe": niqe, "brisque": brisque, "ruido_estimado_sigma": ruido_estimado, "snr_estimado_db": snr_estimado, "curtosis": curtosis, "intensidad_media": intensidad_media, "contraste": contraste})
        else:
            print(f"No se pudo leer {ruta_frame.name}")

df_metricas = pd.DataFrame(resultados)  # Organiza las métricas por frame
columnas_metricas = ["niqe", "brisque", "ruido_estimado_sigma", "snr_estimado_db", "curtosis", "intensidad_media", "contraste"]  # Define las métricas que se resumirán

resumen_metricas = df_metricas[columnas_metricas].describe().T.reset_index()  # Calcula la estadística descriptiva de la etapa
resumen_metricas = resumen_metricas.rename(columns={"index": "metrica", "count": "cantidad", "mean": "media", "std": "desviacion_estandar", "min": "minimo", "25%": "percentil_25", "50%": "mediana", "75%": "percentil_75", "max": "maximo"})

with pd.ExcelWriter(ruta_excel_metricas, engine="openpyxl") as escritor:  # Guarda las métricas detalladas y su estadística descriptiva
    df_metricas.to_excel(escritor, sheet_name="metricas_por_frame", index=False)
    resumen_metricas.to_excel(escritor, sheet_name="estadistica_descriptiva", index=False)

resumen_etapa = df_metricas[columnas_metricas].mean().rename(ETAPA)  # Calcula el valor global de cada métrica para la etapa

if RUTA_EXCEL_RESUMEN.exists():
    resumen_global = pd.read_excel(RUTA_EXCEL_RESUMEN).set_index("metrica")  # Carga las etapas calculadas anteriormente
    resumen_global = resumen_global.reindex(resumen_global.index.union(resumen_etapa.index, sort=False))  # Incorpora métricas nuevas sin eliminar las existentes
    resumen_global[ETAPA] = resumen_etapa  # Agrega o actualiza la columna de la etapa actual
else:
    resumen_global = resumen_etapa.to_frame()  # Crea el resumen cuando todavía no existen otras etapas

resumen_global.index.name = "metrica"  # Nombra la columna que contiene las métricas
resumen_global.reset_index().to_excel(RUTA_EXCEL_RESUMEN, index=False)  # Guarda la comparación global entre etapas

display(resumen_metricas)  # Muestra la estadística descriptiva en el notebook
display(resumen_global.reset_index())  # Muestra la comparación global entre etapas

valores_ruido = np.concatenate(muestras_ruido)  # Une las muestras de ruido de todos los frames
valores_intensidad = np.concatenate(muestras_intensidad)  # Une las muestras de intensidad de todos los frames
media_ruido = float(valores_ruido.mean())
desviacion_ruido = float(valores_ruido.std())
limite_ruido = float(np.percentile(np.abs(valores_ruido), 99.5))  # Limita la gráfica al rango que contiene el 99.5 % de las magnitudes
eje_normal = np.linspace(-limite_ruido, limite_ruido, 300)  # Crea los valores usados para representar la distribución normal
densidad_normal = stats.norm.pdf(eje_normal, loc=media_ruido, scale=desviacion_ruido)  # Calcula la función de densidad de una normal equivalente

figura, eje = plt.subplots(figsize=(6, 4))  # Crea la gráfica de NIQE
sns.histplot(data=df_metricas, x="niqe", bins=30, kde=True, ax=eje)
eje.set_title("Distribución de NIQE")
eje.set_xlabel("NIQE (↓)")
eje.set_ylabel("Cantidad de frames")
figura.tight_layout()  # Ajusta los márgenes al contenido
figura.savefig(str(CARPETA_METRICAS / f"niqe_{ETAPA}.png"), dpi=300, bbox_inches="tight")  # Guarda la gráfica con alta resolución
plt.show()

figura, eje = plt.subplots(figsize=(6, 4))  # Crea la gráfica de BRISQUE
sns.histplot(data=df_metricas, x="brisque", bins=30, kde=True, ax=eje)
eje.set_title("Distribución de BRISQUE")
eje.set_xlabel("BRISQUE (↓)")
eje.set_ylabel("Cantidad de frames")
figura.tight_layout()  # Ajusta los márgenes al contenido
figura.savefig(str(CARPETA_METRICAS / f"brisque_{ETAPA}.png"), dpi=300, bbox_inches="tight")  # Guarda la gráfica con alta resolución
plt.show()

figura, eje = plt.subplots(figsize=(6, 4))  # Crea la gráfica del ruido aproximado
sns.histplot(valores_ruido, bins=100, stat="density", label="Ruido aproximado", ax=eje)
eje.plot(eje_normal, densidad_normal, label="Distribución normal")
eje.set_xlim(-limite_ruido, limite_ruido)
eje.set_title("Distribución del ruido aproximado")
eje.set_xlabel("Valor de la diferencia")
eje.set_ylabel("Densidad")
eje.legend()
figura.tight_layout()  # Ajusta los márgenes al contenido
figura.savefig(str(CARPETA_METRICAS / f"ruido_{ETAPA}.png"), dpi=300, bbox_inches="tight")  # Guarda la gráfica con alta resolución
plt.show()

figura, eje = plt.subplots(figsize=(6, 4))  # Crea la gráfica global de intensidades
sns.histplot(valores_intensidad, bins=256, binrange=(0, 255), ax=eje)
eje.set_xlim(0, 255)
eje.set_title("Distribución global de intensidades")
eje.set_xlabel("Intensidad")
eje.set_ylabel("Cantidad de píxeles")
figura.tight_layout()  # Ajusta los márgenes al contenido
figura.savefig(str(CARPETA_METRICAS / f"intensidad_{ETAPA}.png"), dpi=300, bbox_inches="tight")  # Guarda la gráfica con alta resolución
plt.show()

print(f"Excel de métricas guardado en: {ruta_excel_metricas}")
print(f"Resumen de etapas guardado en: {RUTA_EXCEL_RESUMEN}")
print(f"Gráficas guardadas en: {CARPETA_METRICAS}")
print(f"Frames analizados: {len(df_metricas)}")
print(f"Píxeles usados en la gráfica de ruido: {len(valores_ruido)}")
print(f"Píxeles usados en la gráfica de intensidad: {len(valores_intensidad)}")

Calculando métricas:  26%|██▋       | 873/3300 [03:58<12:02,  3.36frame/s]

# 3. Limpieza HUD
## 3.1. Segmentación CrossHair

In [30]:
CARPETA_MASKS_CROSSHAIR.mkdir(parents=True, exist_ok=True)  # Crea la carpeta de máscaras
CARPETA_DIAGNOSTICOS.mkdir(parents=True, exist_ok=True)  # Crea la carpeta temporal de diagnósticos

for ruta_mask in CARPETA_MASKS_CROSSHAIR.glob("*.png"):
    ruta_mask.unlink()  # Elimina una máscara de pruebas anteriores

for ruta_diagnostico in CARPETA_DIAGNOSTICOS.glob("*.png"):
    ruta_diagnostico.unlink()  # Elimina un diagnóstico de pruebas anteriores

print("Máscaras y diagnósticos anteriores eliminados")

Máscaras y diagnósticos anteriores eliminados


In [34]:
CARPETA_MASKS_CROSSHAIR.mkdir(parents=True, exist_ok=True)  # Crea la carpeta de máscaras

if GENERAR_DIAGNOSTICOS:
    CARPETA_DIAGNOSTICOS.mkdir(parents=True, exist_ok=True)  # Crea la carpeta temporal de diagnósticos

for ruta_mask in CARPETA_MASKS_CROSSHAIR.glob("*.png"):
    ruta_mask.unlink()  # Elimina una máscara de pruebas anteriores

for ruta_diagnostico in CARPETA_DIAGNOSTICOS.glob("*.png"):
    ruta_diagnostico.unlink()  # Elimina un diagnóstico de pruebas anteriores

for indice in tqdm(range(CANTIDAD_FRAMES_DIAGNOSTICO), desc="Diagnosticando crosshair", unit="frame"):
    ruta_frame = CARPETA_FRAMES / f"{indice:04d}.png"
    frame_bgr = cv2.imread(str(ruta_frame), cv2.IMREAD_COLOR)  # Lee el frame

    if frame_bgr is not None:
        alto, ancho = frame_bgr.shape[:2]
        ancho_roi = int(ancho * PORCENTAJE_ROI_CROSSHAIR)
        alto_roi = int(alto * PORCENTAJE_ROI_CROSSHAIR)
        x_inicial = (ancho - ancho_roi) // 2
        y_inicial = (alto - alto_roi) // 2
        x_final = x_inicial + ancho_roi
        y_final = y_inicial + alto_roi
        roi_bgr = frame_bgr[y_inicial:y_final, x_inicial:x_final]  # Extrae la zona central donde puede aparecer el crosshair

        roi_hsv = cv2.cvtColor(roi_bgr, cv2.COLOR_BGR2HSV)  # Separa tono, saturación y brillo
        azul, verde, rojo = cv2.split(roi_bgr)
        azul = azul.astype(np.int16)
        verde = verde.astype(np.int16)
        rojo = rojo.astype(np.int16)

        mascara_hsv_fuerte = cv2.inRange(roi_hsv, np.array(HSV_VERDE_FUERTE_MIN, dtype=np.uint8), np.array(HSV_VERDE_FUERTE_MAX, dtype=np.uint8))  # Detecta el núcleo fluorescente del HUD
        domina_fuerte = ((verde >= rojo + DOMINANCIA_VERDE_FUERTE[0]) & (verde >= azul + DOMINANCIA_VERDE_FUERTE[1])).astype(np.uint8) * 255  # Exige una dominancia verde clara
        mascara_fuerte = cv2.bitwise_and(mascara_hsv_fuerte, domina_fuerte)  # Combina el rango HSV con la dominancia verde

        mascara_hsv_borde = cv2.inRange(roi_hsv, np.array(HSV_VERDE_BORDE_MIN, dtype=np.uint8), np.array(HSV_VERDE_BORDE_MAX, dtype=np.uint8))  # Detecta verdes debilitados por transparencia
        domina_borde = ((verde >= rojo + DOMINANCIA_VERDE_BORDE[0]) & (verde >= azul + DOMINANCIA_VERDE_BORDE[1])).astype(np.uint8) * 255  # Permite una dominancia menor en los bordes
        mascara_bordes = cv2.bitwise_and(mascara_hsv_borde, domina_borde)  # Obtiene candidatos para los bordes transparentes

        mascara_color = mascara_fuerte.copy()
        kernel_vecindad = cv2.getStructuringElement(cv2.MORPH_CROSS, (3, 3))  # Define la conexión directa entre píxeles vecinos

        for _ in range(RADIO_BORDES_CROSSHAIR):
            mascara_expandida = cv2.dilate(mascara_color, kernel_vecindad, iterations=1)  # Busca píxeles débiles alrededor del verde confirmado
            nuevos_bordes = cv2.bitwise_and(mascara_expandida, mascara_bordes)  # Conserva únicamente los bordes conectados al núcleo
            mascara_color = cv2.bitwise_or(mascara_color, nuevos_bordes)  # Añade los bordes transparentes compatibles

        mascara_color = cv2.morphologyEx(mascara_color, cv2.MORPH_CLOSE, kernel_vecindad, iterations=1)  # Repara interrupciones aisladas de un píxel

        dimension_roi = min(ancho_roi, alto_roi)
        longitud_minima = max(7, int(dimension_roi * 0.015))
        longitud_maxima = max(longitud_minima + 2, int(dimension_roi * 0.055))
        grosor_plantilla = max(1, int(dimension_roi * 0.003))
        candidatos_esquinas = []

        for longitud in range(longitud_minima, longitud_maxima + 1, max(2, grosor_plantilla)):
            margen = grosor_plantilla + 1
            tamano_plantilla = longitud + margen * 2
            plantillas = {}

            plantilla = np.zeros((tamano_plantilla, tamano_plantilla), dtype=np.uint8)
            cv2.line(plantilla, (margen, margen), (margen + longitud, margen), 255, grosor_plantilla)
            cv2.line(plantilla, (margen, margen), (margen, margen + longitud), 255, grosor_plantilla)
            plantillas["superior_izquierda"] = (plantilla, (margen, margen))

            plantilla = np.zeros((tamano_plantilla, tamano_plantilla), dtype=np.uint8)
            cv2.line(plantilla, (margen + longitud, margen), (margen, margen), 255, grosor_plantilla)
            cv2.line(plantilla, (margen + longitud, margen), (margen + longitud, margen + longitud), 255, grosor_plantilla)
            plantillas["superior_derecha"] = (plantilla, (margen + longitud, margen))

            plantilla = np.zeros((tamano_plantilla, tamano_plantilla), dtype=np.uint8)
            cv2.line(plantilla, (margen, margen + longitud), (margen + longitud, margen + longitud), 255, grosor_plantilla)
            cv2.line(plantilla, (margen, margen + longitud), (margen, margen), 255, grosor_plantilla)
            plantillas["inferior_izquierda"] = (plantilla, (margen, margen + longitud))

            plantilla = np.zeros((tamano_plantilla, tamano_plantilla), dtype=np.uint8)
            cv2.line(plantilla, (margen + longitud, margen + longitud), (margen, margen + longitud), 255, grosor_plantilla)
            cv2.line(plantilla, (margen + longitud, margen + longitud), (margen + longitud, margen), 255, grosor_plantilla)
            plantillas["inferior_derecha"] = (plantilla, (margen + longitud, margen + longitud))

            for orientacion, datos_plantilla in plantillas.items():
                plantilla, posicion_vertice = datos_plantilla

                if mascara_color.shape[0] >= plantilla.shape[0] and mascara_color.shape[1] >= plantilla.shapecoincidencia = cv2.matchTemplate(mascara_color, plantilla, cv2.TM_CCOEFF_NORMED)  # Compara la máscara con una esquina recta
                    _, puntuacion, _, posicion = cv2.minMaxLoc(coincidencia)  # Obtiene la coincidencia más alta y su posición

                    if puntuacion >= 0.42:
                        vertice_x = posicion[0] + posicion_vertice[0]
                        vertice_y = posicion[1] + posicion_vertice[1]
                        candidatos_esquinas.append({"orientacion": orientacion, "x": vertice_x, "y": vertice_y, "longitud": longitud, "puntuacion": float(puntuacion)})  # Guarda la posible esquina

        candidatos_esquinas = sorted(candidatos_esquinas, key=lambda candidato: candidato["puntuacion"], reverse=True)  # Ordena los candidatos desde la mayor coincidencia
        candidatos_filtrados = []

        for candidato in candidatos_esquinas:
            distancia_minima = max(5, candidato["longitud"] // 2)
            candidatos_misma_orientacion = [existente for existente in candidatos_filtrados if existente["orientacion"] == candidato["orientacion"]]
            candidato_repetido = any(np.hypot(candidato["x"] - existente["x"], candidato["y"] - existente["y"]) < distancia_minima for existente in candidatos_misma_orientacion)  # Comprueba si la esquina ya fue detectada en otra escala

            if not candidato_repetido and len(candidatos_misma_orientacion) < 4:
                candidatos_filtrados.append(candidato)  # Conserva candidatos diferentes y limita repeticiones por orientación

        mejor_pareja = None
        mejor_puntuacion_pareja = -np.inf
        parejas_opuestas = [("superior_izquierda", "inferior_derecha"), ("superior_derecha", "inferior_izquierda")]

        for orientacion_a, orientacion_b in parejas_opuestas:
            esquinas_a = [candidato for candidato in candidatos_filtrados if candidato["orientacion"] == orientacion_a]
            esquinas_b = [candidato for candidato in candidatos_filtrados if candidato["orientacion"] == orientacion_b]

            for esquina_a in esquinas_a:
                for esquina_b in esquinas_b:
                    diferencia_x = abs(esquina_b["x"] - esquina_a["x"])
                    diferencia_y = abs(esquina_b["y"] - esquina_a["y"])
                    proporcion = diferencia_x / max(diferencia_y, 1)
                    escalas_compatibles = min(esquina_a["longitud"], esquina_b["longitud"]) / max(esquina_a["longitud"], esquina_b["longitud"])
                    centro_x = int(round((esquina_a["x"] + esquina_b["x"]) / 2))
                    centro_y = int(round((esquina_a["y"] + esquina_b["y"]) / 2))
                    distancia_centro = np.hypot(centro_x - ancho_roi / 2, centro_y - alto_roi / 2) / dimension_roi

                    if diferencia_x >= dimension_roi * 0.08 and diferencia_y >= dimension_roi * 0.08 and 0.65 <= proporcion <= 1.55 and escalas_compatibles >= 0.70:
                        puntuacion_pareja = (esquina_a["puntuacion"] + esquina_b["puntuacion"]) / 2 + escalas_compatibles * 0.20 - distancia_centro * 0.15  # Combina coincidencia, escala y posición aproximada

                        if puntuacion_pareja > mejor_puntuacion_pareja:
                            mejor_puntuacion_pareja = puntuacion_pareja
                            mejor_pareja = (esquina_a, esquina_b, centro_x, centro_y)

        mascara_completa = np.zeros((alto, ancho), dtype=np.uint8)
        mascara_completa[y_inicial:y_final, x_inicial:x_final] = mascara_color  # Coloca la máscara refinada en el tamaño completo del frame
        cv2.imwrite(str(CARPETA_MASKS_CROSSHAIR / ruta_frame.name), mascara_completa, [cv2.IMWRITE_PNG_COMPRESSION, COMPRESION_PNG])  # Guarda únicamente la máscara de color

        if GENERAR_DIAGNOSTICOS:
            diagnostico = frame_bgr.copy()
            capa_roja = frame_bgr.copy()
            capa_roja[mascara_completa > 0] = (0, 0, 255)  # Colorea en rojo la máscara refinada
            superposicion = cv2.addWeighted(frame_bgr, 0.45, capa_roja, 0.55, 0)  # Combina el frame con la capa roja
            diagnostico[mascara_completa > 0] = superposicion[mascara_completa > 0]  # Aplica la superposición únicamente sobre la máscara
            cv2.rectangle(diagnostico, (x_inicial, y_inicial), (x_final - 1, y_final - 1), (255, 0, 0), 2)  # Marca en azul la ROI evaluada

            for candidato in candidatos_filtrados:
                punto = (x_inicial + candidato["x"], y_inicial + candidato["y"])
                cv2.circle(diagnostico, punto, max(3, grosor_plantilla * 2), (0, 255, 255), 2)  # Marca en amarillo una posible esquina

            if mejor_pareja is not None and mejor_puntuacion_pareja >= 0.47:
                esquina_a, esquina_b, centro_x, centro_y = mejor_pareja
                punto_a = (x_inicial + esquina_a["x"], y_inicial + esquina_a["y"])
                punto_b = (x_inicial + esquina_b["x"], y_inicial + esquina_b["y"])
                punto_centro = (x_inicial + centro_x, y_inicial + centro_y)
                cv2.circle(diagnostico, punto_a, max(5, grosor_plantilla * 3), (255, 255, 0), 2)  # Marca en cian la primera esquina aceptada
                cv2.circle(diagnostico, punto_b, max(5, grosor_plantilla * 3), (255, 255, 0), 2)  # Marca en cian la esquina diagonal opuesta
                cv2.line(diagnostico, punto_a, punto_b, (255, 255, 0), 1)  # Une la pareja usada para inferir el centro
                cv2.drawMarker(diagnostico, punto_centro, (255, 0, 255), cv2.MARKER_CROSS, max(12, grosor_plantilla * 6), 2)  # Marca en magenta el centro inferido
                cv2.putText(diagnostico, f"Pareja aceptada: {mejor_puntuacion_pareja:.2f}", (x_inicial + 10, y_inicial + 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2)  # Muestra la confianza de la pareja
            else:
                cv2.putText(diagnostico, "Sin pareja opuesta confiable", (x_inicial + 10, y_inicial + 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 165, 255), 2)  # Indica que no se infirió una geometría

            cv2.imwrite(str(CARPETA_DIAGNOSTICOS / ruta_frame.name), diagnostico, [cv2.IMWRITE_PNG_COMPRESSION, COMPRESION_PNG])  # Guarda el diagnóstico sin reconstruir el crosshair
    else:
        print(f"No se pudo leer {ruta_frame.name}")

print(f"Primeros {CANTIDAD_FRAMES_DIAGNOSTICO} frames evaluados")
print(f"Máscaras de color guardadas en: {CARPETA_MASKS_CROSSHAIR}")

if GENERAR_DIAGNOSTICOS:
    print(f"Diagnósticos guardados en: {CARPETA_DIAGNOSTICOS}")

SyntaxError: invalid syntax (3980100069.py, line 84)